In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
from pathlib import Path

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE_PATH = Path('/content/drive/MyDrive/AI/GMAv3/best_models')

In [4]:
model_names = list(map(lambda x: x.name, BASE_PATH.iterdir()))

In [5]:
from transformers import AutoConfig, AutoModel, AutoTokenizer
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'roberta-base'

class GMAv3(nn.Module):
    def __init__(self, num_classes=6):
        super(GMAv3, self).__init__()
        self.huggingface_config = AutoConfig.from_pretrained(model_name)
        self.huggingface = AutoModel.from_config(self.huggingface_config)
        self.fc = nn.Linear(self.huggingface_config.hidden_size, num_classes)

    def forward(self, x, att_masks):
        x = self.huggingface(x, att_masks)
        x = x.last_hidden_state[:, 0, :]
        x = self.fc(x)
        return x


def load_model(model_name):
    state_dict = torch.load(BASE_PATH / model_name, map_location=DEVICE, weights_only=True)['model_state_dict']
    model = GMAv3()
    model.load_state_dict(state_dict)
    return model

In [6]:
import json
author_dict = {}
with open(BASE_PATH / '../author_dict.json', 'r') as f:
    author_dict = json.load(f)

authors = list(author_dict.values())

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
models = list(map(load_model, model_names))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
from tabulate import tabulate
from termcolor import colored

message = input("Message: ")
ids = tokenizer.encode(message, padding=True, truncation=False, return_tensors='pt')
att_masks = torch.ones_like(ids)

preds = []
for model in models:
    pred = model(ids, att_masks)[0]
    pred = F.softmax(pred, dim=0)
    preds.append(pred)

preds = np.array([t.detach().numpy() for t in preds]).T
avgs = preds.mean(axis=1)
preds = np.concatenate((preds, avgs.reshape(-1, 1)), axis=1)
preds = preds.round(3)
preds = preds.tolist()

headers = [f'model_{i}' for i in range(1, 5)] + ['average']
authors_censored = [f"{s[0]}{(len(s) - 2) * '*'}{s[-1]}" for s in authors]
row_names = authors_censored

print(tabulate(preds, headers=headers, tablefmt='grid', showindex=row_names))

Message: between #2 and #3 I got so insanely lucky
+----------+-----------+-----------+-----------+-----------+-----------+
|          |   model_1 |   model_2 |   model_3 |   model_4 |   average |
+==========+===========+===========+===========+===========+===========+
| k****r   |     0.002 |     0.001 |     0.001 |     0.001 |     0.001 |
+----------+-----------+-----------+-----------+-----------+-----------+
| k****n   |     0.003 |     0.002 |     0.012 |     0.001 |     0.004 |
+----------+-----------+-----------+-----------+-----------+-----------+
| l*****y  |     0.871 |     0.966 |     0.938 |     0.979 |     0.939 |
+----------+-----------+-----------+-----------+-----------+-----------+
| o******0 |     0.118 |     0.028 |     0.019 |     0.016 |     0.045 |
+----------+-----------+-----------+-----------+-----------+-----------+
| p*****_  |     0.001 |     0     |     0     |     0.001 |     0.001 |
+----------+-----------+-----------+-----------+-----------+-----------+
